# GPU で音声を文字起こし

[Whisper small](https://huggingface.co/openai/whisper-small) で日本語を含む音声を文字起こしし、TXT と SRT 字幕を ZIP にまとめます。モデルは Apache-2.0 ライセンスです。音声ファイルは Colab の一時領域で処理されます。初回はモデルをダウンロードします。

**ランタイム → ランタイムのタイプを変更 → GPU** を選び、上から順に実行してください。ファイルの内容を共有する際は個人情報を確認してください。

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU が見つかりません。Colab の「ランタイム → ランタイムのタイプを変更」で GPU を選んでから再実行してください。')
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)
print('空きVRAM: %.1f GiB' % (torch.cuda.mem_get_info()[0] / 1024**3))


## ライブラリを準備

In [ ]:
%pip -q install "transformers>=4.50,<5" accelerate soundfile


## 音声を1つアップロード

MP3 / WAV / M4A / FLAC / OGG / WEBM に対応します。長いファイルは処理に時間がかかります。アップロード上限は 100 MiB です。

In [ ]:
from pathlib import Path
from google.colab import files
import shutil
import tempfile

if not shutil.which('ffmpeg'):
    raise RuntimeError('ffmpeg がありません。Colab ランタイムで実行してください。')
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('音声ファイルを1つだけ選択してください。')
name, data = next(iter(uploaded.items()))
safe_name = Path(name).name
if Path(safe_name).suffix.lower() not in {'.mp3', '.wav', '.m4a', '.flac', '.ogg', '.webm'}:
    raise ValueError('対応形式: MP3 / WAV / M4A / FLAC / OGG / WEBM')
if len(data) > 100 * 1024**2:
    raise ValueError('ファイルが 100 MiB を超えています。短く分割してください。')
work_dir = Path(tempfile.mkdtemp(prefix='whisper_', dir='/content'))
audio_path = work_dir / ('input' + Path(safe_name).suffix.lower())
audio_path.write_bytes(data)
print('入力:', safe_name, 'サイズ:', round(len(data) / 1024**2, 1), 'MiB')


## 推論する

`LANGUAGE = None` なら言語を自動判別します。日本語と分かっている音声なら `LANGUAGE = "japanese"` にできます。曲の歌詞や雑音の多い音声では誤認識に注意してください。

In [ ]:
from transformers import pipeline

MODEL_ID = 'openai/whisper-small'
LANGUAGE = None  # 例: 'japanese' / 'english' / None（自動判別）
asr = pipeline('automatic-speech-recognition', model=MODEL_ID,
               device=0, torch_dtype=torch.float16, chunk_length_s=30)
generate_kwargs = {'task': 'transcribe'}
if LANGUAGE:
    generate_kwargs['language'] = LANGUAGE
result = asr(str(audio_path), return_timestamps=True, generate_kwargs=generate_kwargs)
print(result['text'])


## TXT・SRT を保存してダウンロード

タイムスタンプはモデルの推定値です。字幕を公開する前に音声と照合してください。

In [ ]:
import zipfile

def srt_time(seconds):
    ms = max(0, round(float(seconds) * 1000))
    hours, ms = divmod(ms, 3_600_000)
    minutes, ms = divmod(ms, 60_000)
    secs, ms = divmod(ms, 1_000)
    return f'{hours:02}:{minutes:02}:{secs:02},{ms:03}'

text_path = work_dir / 'transcript.txt'
text_path.write_text(result['text'].strip() + '\n', encoding='utf-8')
srt_lines = []
for index, chunk in enumerate(result.get('chunks', []), start=1):
    start, end = chunk['timestamp']
    if start is None:
        continue
    end = max(float(start) + 0.01, float(end)) if end is not None else float(start) + 0.01
    srt_lines.extend([str(index), f'{srt_time(start)} --> {srt_time(end)}', chunk['text'].strip(), ''])
srt_path = work_dir / 'transcript.srt'
srt_path.write_text('\n'.join(srt_lines), encoding='utf-8')
zip_path = work_dir / 'transcript.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(text_path, arcname=text_path.name)
    archive.write(srt_path, arcname=srt_path.name)
files.download(str(zip_path))
